# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hussaintinwala2/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Install required packages
!pip -q install duckdb huggingface_hub

In [2]:
import os
import duckdb
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()

# Authenticate DuckDB with Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Warehouse location
REL = "hf://datasets/FlyRank/internship-warehouse"

# Tables we will use
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected successfully.")

DuckDB connected successfully.


In [3]:
# Quick connection test
print("Daily rows:", con.sql(
    f"SELECT COUNT(*) FROM {TABLES['fact_daily']}"
).fetchone()[0])

print("Connection is working.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Daily rows: 78835655
Connection is working.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### My contract

**Unit of analysis:** One content page for one client.

For the development slice, I will use content performance data from March 2026. Daily performance records will be aggregated to the content-page level so that each final row represents one page for one client over the selected time window.

The goal is to use information available before the decision point to help rank pages by their priority for content review or refresh.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the grain of the daily performance table for March 2026

grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate rows at client + content + date grain:")
print(grain_check)

if len(grain_check) == 0:
    print("\nVerified: one row represents one client × content page × day.")
else:
    print("\nWarning: duplicate rows were found.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows at client + content + date grain:
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, row_count]
Index: []

Verified: one row represents one client × content page × day.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

My field contract

Features — information available at the decision moment:

gsc_impressions — observed search visibility of the page during the feature window.
gsc_clicks — observed clicks received from search during the feature window.
gsc_avg_position — observed average search position during the feature window.
ga4_sessions — observed sessions during the feature window when GA4 data is available.
sessions_ai — observed AI-attributed sessions during the feature window when the relevant data is available.

Label / outcome:

Future change in impressions after the decision point. The label will be calculated from a later observation window using the observed change in impressions. It will not use information from the future window as a feature.

Context:

client_hash_id — identifies the client and can be used for grouping and client-level train/test splitting, but not as a predictive feature.
content_hash_id — identifies the content page and is used to keep track of the unit of analysis, but not as a predictive feature.
report_date — identifies when the observation was recorded and is used to define the feature and outcome windows.

Excluded:

client_hash_id and content_hash_id — excluded as predictive features because they are identifiers rather than meaningful page characteristics.
report_date — excluded as a predictive feature because it defines the observation window rather than describing the content itself.
ga4_sessions and sessions_ai will be excluded from any model or comparison where the required GA4 data is not available, rather than treating missing values as observed zero activity.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

> Add blockquote



*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: Verify the grain
# Expected grain: one row per client + content + day

grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate client × content × day combinations:")
print(grain_check)

if len(grain_check) == 0:
    print("\nVerified: the daily table has one row per client × content × day.")
else:
    print("\nWarning: duplicate combinations were found.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate client × content × day combinations:
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, row_count]
Index: []

Verified: the daily table has one row per client × content × day.


In [9]:
# Query 2: Row count and date span for March 2026

march_stats = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

print(march_stats)

   row_count first_date  last_date
0    9841378 2026-03-01 2026-03-31


In [10]:
# Query 3: Check GA4 availability

availability_check = con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY ga4_data_available
    ORDER BY ga4_data_available
""").df()

print("GA4 availability in March 2026:")
print(availability_check)

usable_rows = con.sql(f"""
    SELECT COUNT(*) AS usable_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND ga4_data_available IS TRUE
""").fetchone()[0]

print(f"\nRows with GA4 data available: {usable_rows:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GA4 availability in March 2026:
   ga4_data_available  row_count
0               False    6408671
1                True     413966
2                <NA>    3018741

Rows with GA4 data available: 413,966


In [12]:
# Inspect the columns in the daily performance table

schema = con.sql(f"""
    SELECT *
    FROM {TABLES['fact_daily']}
    LIMIT 0
""").df()

print("Columns in fact_daily:")
print(schema.columns.tolist())

Columns in fact_daily:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits

This slice has an unbalanced history across clients, so not every client has the same amount of historical data. GA4 is also unavailable for many rows, so GA4-based features cannot be treated as universally available. The March development window can also overlap with future outcome windows, so feature windows and label windows must be separated to avoid leakage. Therefore, results from this slice should be treated as directional decision-support rather than a complete measure of content performance.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Check important data limitations

# 1. Check the range of history across clients
client_history = con.sql(f"""
    SELECT
        client_hash_id,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT month) AS months_available
    FROM {TABLES['fact_daily']}
    GROUP BY client_hash_id
""").df()

print("Client history summary:")
print(client_history["months_available"].describe())


# 2. Check GSC and GA4 availability in March 2026
availability_summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

print("\nMarch 2026 availability:")
print(availability_summary)


# 3. Show the March date range used for this contract
window_check = con.sql(f"""
    SELECT
        MIN(report_date) AS feature_window_start,
        MAX(report_date) AS feature_window_end,
        COUNT(DISTINCT report_date) AS number_of_days
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

print("\nMarch 2026 window:")
print(window_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Client history summary:
count    70.000000
mean      7.714286
std       4.121850
min       2.000000
25%       4.000000
50%       8.000000
75%      10.000000
max      18.000000
Name: months_available, dtype: float64


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


March 2026 availability:
   total_rows  gsc_available  ga4_available
0     9841378        3611061         413966

March 2026 window:
  feature_window_start feature_window_end  number_of_days
0           2026-03-01         2026-03-31              31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.